# Alethe — Observability Watermark Specification

Progressive testing across four stages:

| Stage | File | Infrastructure |
|---|---|---|
| 0 — Algebraic foundation | `obs_semiring.py` | None (pure Python) |
| 1 — Simulation grade | `ows_examples.py` | None (fake infra, real logic) |
| 2 — Real Delta Lake | `ows_delta_oracle.py` | Real Delta table + VACUUM |
| 3 — Manifest + Iceberg | `ows_manifest_and_iceberg.py` | Real Iceberg + manifest |

**Dependencies:**
```
pip install deltalake pyiceberg pyarrow pandas
```

> Phases 2 and 3 write real tables to `./lakehouse/` and `./iceberg_warehouse/` relative to this notebook.

---
## Phase 0: The Algebraic Foundation — Observability Semiring

The core claim of OWS: **unknowability should be a value, not an error.** This phase defines the three-valued semiring `K = {ABSENT=0, BEYOND=⊥, OBSERVED=1}` and proves all 126 semiring identities hold exhaustively.

The payoff: once `BEYOND` is a proper semiring element, refusal propagates through any relational operator (select, project, join, union) *by algebra alone* — no special-case code anywhere in the query engine.

### 0.1 The semiring: `K`, `⊕`, `⊗`

`K.BEYOND` is *not* SQL NULL. NULL means "a value exists but is unknown"; `BEYOND` means "the question exceeds a retention boundary." The difference matters: NULLs silently vanish from joins; `BEYOND` rows are *candidates* — they keep their join keys and taint every row they touch via `⊗ = min`.

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from enum import IntEnum
from itertools import product
from typing import Callable, Hashable


class K(IntEnum):
    ABSENT = 0    # additive identity (semiring "0")
    BEYOND = 1    # the refusal value
    OBSERVED = 2  # multiplicative identity (semiring "1")

    def __repr__(self) -> str:
        return self.name


def k_add(a: K, b: K) -> K:
    """⊕ : union / alternative derivations. OBSERVED absorbs; BEYOND beats ABSENT."""
    return K(max(a, b))


def k_mul(a: K, b: K) -> K:
    """⊗ : join / conjunction. ABSENT annihilates; BEYOND taints OBSERVED."""
    return K(min(a, b))


ZERO, ONE = K.ABSENT, K.OBSERVED

### 0.2 Truth tables for `⊕` (union) and `⊗` (join)

In [2]:
from itertools import product

print("⊕ (union / add / max):")
print(f"  {'':8}", end="")
for b in K:
    print(f"{b.name:10}", end="")
print()
for a in K:
    print(f"  {a.name:8}", end="")
    for b in K:
        print(f"{k_add(a, b).name:10}", end="")
    print()

print()
print("⊗ (join / mul / min):")
print(f"  {'':8}", end="")
for b in K:
    print(f"{b.name:10}", end="")
print()
for a in K:
    print(f"  {a.name:8}", end="")
    for b in K:
        print(f"{k_mul(a, b).name:10}", end="")
    print()

⊕ (union / add / max):
          ABSENT    BEYOND    OBSERVED  
  ABSENT  ABSENT    BEYOND    OBSERVED  
  BEYOND  BEYOND    BEYOND    OBSERVED  
  OBSERVEDOBSERVED  OBSERVED  OBSERVED  

⊗ (join / mul / min):
          ABSENT    BEYOND    OBSERVED  
  ABSENT  ABSENT    ABSENT    ABSENT    
  BEYOND  ABSENT    BEYOND    BEYOND    
  OBSERVEDABSENT    BEYOND    OBSERVED  


### 0.3 Exhaustive semiring law verification

All 126 identities (associativity, commutativity, distributivity, identity/annihilator elements) are machine-checked over the 3-element carrier. This is the spec requirement — conforming implementations must verify these.

In [3]:
def verify_semiring_laws() -> list[str]:
    """Exhaustively check all semiring axioms over the 3-element carrier."""
    failures = []
    E = list(K)
    for a, b, c in product(E, repeat=3):
        if k_add(k_add(a, b), c) != k_add(a, k_add(b, c)):
            failures.append(f"⊕ not associative at {a},{b},{c}")
        if k_mul(k_mul(a, b), c) != k_mul(a, k_mul(b, c)):
            failures.append(f"⊗ not associative at {a},{b},{c}")
        if k_mul(a, k_add(b, c)) != k_add(k_mul(a, b), k_mul(a, c)):
            failures.append(f"left distributivity fails at {a},{b},{c}")
        if k_mul(k_add(a, b), c) != k_add(k_mul(a, c), k_mul(b, c)):
            failures.append(f"right distributivity fails at {a},{b},{c}")
    for a, b in product(E, repeat=2):
        if k_add(a, b) != k_add(b, a):
            failures.append(f"⊕ not commutative at {a},{b}")
    for a in E:
        if k_add(a, ZERO) != a:
            failures.append(f"0 not additive identity at {a}")
        if k_mul(a, ONE) != a or k_mul(ONE, a) != a:
            failures.append(f"1 not multiplicative identity at {a}")
        if k_mul(a, ZERO) != ZERO or k_mul(ZERO, a) != ZERO:
            failures.append(f"0 not annihilating at {a}")
    return failures

In [4]:
failures = verify_semiring_laws()
n_checked = 27 * 4 + 9 + 3 * 3   # triples×4 + comm pairs + identity/annihilator
print(f"Identities checked: {n_checked}")
print(f"Failures: {failures if failures else 'none — ALL LAWS HOLD'}")

Identities checked: 126
Failures: none — ALL LAWS HOLD


### 0.4 K-Relations

A K-relation maps each row to a `K` annotation. Relational operators are defined purely by the semiring laws — no special refusal handling needed:- `project` merges duplicates with `⊕`
- `join` multiplies annotations with `⊗`
- `union` adds annotations with `⊕`

In [5]:
Row = tuple  # immutable tuple of column values


@dataclass
class KRelation:
    """A K-annotated relation: mapping from rows to K values.

    Rows mapping to ABSENT are simply not stored (they're the implicit
    default), matching the standard K-relation formalism where the
    annotation function has finite support.
    """
    columns: tuple[str, ...]
    data: dict[Row, K] = field(default_factory=dict)

    def add(self, row: Row, ann: K) -> None:
        if ann == K.ABSENT:
            return
        existing = self.data.get(row, K.ABSENT)
        self.data[row] = k_add(existing, ann)

    # --- relational operators, all defined purely by semiring laws ---

    def select(self, pred: Callable[[dict], bool]) -> "KRelation":
        out = KRelation(self.columns)
        for row, ann in self.data.items():
            if pred(dict(zip(self.columns, row))):
                out.add(row, ann)
        return out

    def project(self, cols: tuple[str, ...]) -> "KRelation":
        """Projection merges duplicate rows with ⊕ — where refusal
        semantics really earns its keep."""
        idx = [self.columns.index(c) for c in cols]
        out = KRelation(cols)
        for row, ann in self.data.items():
            out.add(tuple(row[i] for i in idx), ann)
        return out

    def join(self, other: "KRelation", on: list[str]) -> "KRelation":
        """Natural join: matching rows multiply annotations with ⊗."""
        out_cols = self.columns + tuple(c for c in other.columns if c not in on)
        out = KRelation(out_cols)
        my_idx = [self.columns.index(c) for c in on]
        their_idx = [other.columns.index(c) for c in on]
        their_rest = [i for i, c in enumerate(other.columns) if c not in on]
        # hash join on the key
        buckets: dict[tuple, list[tuple[Row, K]]] = {}
        for row, ann in other.data.items():
            buckets.setdefault(tuple(row[i] for i in their_idx), []).append((row, ann))
        for row, ann in self.data.items():
            key = tuple(row[i] for i in my_idx)
            for orow, oann in buckets.get(key, []):
                combined = k_mul(ann, oann)
                if combined != K.ABSENT:
                    out.add(row + tuple(orow[i] for i in their_rest), combined)
        return out

    def union(self, other: "KRelation") -> "KRelation":
        out = KRelation(self.columns)
        for row, ann in self.data.items():
            out.add(row, ann)
        for row, ann in other.data.items():
            out.add(row, ann)
        return out

    def show(self, title: str = "") -> None:
        if title:
            print(f"\n  {title}")
        header = " | ".join(f"{c:<14}" for c in self.columns) + " | annotation"
        print("  " + header)
        print("  " + "-" * len(header))
        for row, ann in sorted(self.data.items(), key=lambda x: str(x[0])):
            cells = " | ".join(f"{str(v):<14}" for v in row)
            print(f"  {cells} | {ann!r}")
        if not self.data:
            print("  (empty)")

### 0.5 `TemporalTable.as_of(t)` — retention-aware snapshots

When `t < retention_watermark`: history is gone. Every entity the table has ever known about is annotated `BEYOND` — we can't say what its state was, or even whether it existed at `t`. The *annotation* (not value masking) is what conveys epistemic status; masking join keys would make `BEYOND` rows silently vanish from joins.

In [6]:
@dataclass
class TemporalTable:
    """A bitemporal-ish table: each version of a row carries a system-time
    interval [valid_from, valid_to). The table also carries a retention
    watermark: history before it has been vacuumed/expired.

    `as_of(t)` produces a K-relation:
      - t >= watermark: rows valid at t are OBSERVED. Clean snapshot.
      - t <  watermark: history is gone. Every *entity* this table has
        ever known about is annotated BEYOND — we cannot say what its
        state was, nor even whether it existed at t. The algebra takes
        it from there.
    """
    name: str
    columns: tuple[str, ...]          # logical columns (no temporal cols)
    key_columns: tuple[str, ...]      # entity key
    retention_watermark: int          # earliest system time still observable
    versions: list[tuple[Row, int, int]] = field(default_factory=list)
    INF = 10**9

    def insert_version(self, row: Row, valid_from: int, valid_to: int = INF):
        self.versions.append((row, valid_from, valid_to))

    def as_of(self, t: int) -> KRelation:
        rel = KRelation(self.columns)
        if t >= self.retention_watermark:
            for row, vf, vt in self.versions:
                if vf <= t < vt:
                    rel.add(row, K.OBSERVED)
            return rel
        # Query predates retention. We cannot reconstruct the state at t,
        # but we know which row-shapes this table has ever contained. Emit
        # each distinct known row as a CANDIDATE annotated BEYOND: the
        # annotation — not value masking — is what marks it untrustworthy.
        # (Masking the join key would make BEYOND rows silently vanish
        # from joins, producing exactly the false confidence we're trying
        # to eliminate. Candidates keep the algebra flowing; ⊗ taints
        # everything they touch.)
        seen: set[Row] = set()
        for row, _, _ in self.versions:
            if row not in seen:
                seen.add(row)
                rel.add(row, K.BEYOND)
        return rel


@dataclass
class QueryResult:
    answers: KRelation           # rows annotated OBSERVED
    refusals: KRelation          # rows annotated BEYOND
    refused: bool                # did the query exceed any observability boundary?

    def report(self, title: str) -> None:
        print("\n" + "=" * 72)
        print(f"  QUERY: {title}")
        print("=" * 72)
        self.answers.show("ANSWERS (observed, inside retention):")
        if self.refused:
            self.refusals.show("REFUSALS (beyond observability boundary):")
            print("\n  ⚠ HONEST REFUSAL: parts of this answer are unknowable —")
            print("    the query's time bound precedes a retention watermark.")
            print("    These rows are neither confirmed nor denied.")
        else:
            print("\n  ✓ Complete answer: fully inside all observability boundaries.")


def split_result(rel: KRelation) -> QueryResult:
    """Partition a K-relation into answers vs refusals. This is the ONLY
    place refusal is 'handled' — everything upstream is plain algebra."""
    answers = KRelation(rel.columns)
    refusals = KRelation(rel.columns)
    for row, ann in rel.data.items():
        (answers if ann == K.OBSERVED else refusals).add(row, ann)
    return QueryResult(answers, refusals, refused=bool(refusals.data))


# ---------------------------------------------------------------------------
# 4. Demo: a temporal join across mismatched retention boundaries
# ---------------------------------------------------------------------------

### 0.6 Build sample temporal tables

- `customers`: long retention (back to t=0)
- `orders`: short retention (vacuumed before t=40) — the realistic scenario   where analytical replicas have shorter history than the OLTP source

In [7]:
customers = TemporalTable(
    name="customers",
    columns=("customer_id", "segment"),
    key_columns=("customer_id",),
    retention_watermark=0,
)
customers.insert_version(("C1", "smb"), 0, 50)
customers.insert_version(("C1", "enterprise"), 50)   # upgraded at t=50
customers.insert_version(("C2", "smb"), 10)
customers.insert_version(("C3", "enterprise"), 30)

orders = TemporalTable(
    name="orders",
    columns=("order_id", "customer_id", "amount"),
    key_columns=("order_id",),
    retention_watermark=40,           # history before t=40 was vacuumed
)
orders.insert_version(("O1", "C1", 100), 20)  # created t=20 — pre-watermark!
orders.insert_version(("O2", "C2", 250), 45)
orders.insert_version(("O3", "C1", 900), 60)

print("orders.retention_watermark =", orders.retention_watermark)
print("customers.retention_watermark =", customers.retention_watermark)

orders.retention_watermark = 40
customers.retention_watermark = 0


### 0.7 Query inside both retention boundaries (t=60)

Both tables have history at t=60 — the join is fully `OBSERVED`, verdict is `EXACT`.

In [8]:
c60 = customers.as_of(60)
o60 = orders.as_of(60)
joined60 = o60.join(c60, on=["customer_id"])
result60 = split_result(joined60)
result60.report("orders ⋈ customers AS OF t=60  (orders watermark=40, customers watermark=0)")


  QUERY: orders ⋈ customers AS OF t=60  (orders watermark=40, customers watermark=0)

  ANSWERS (observed, inside retention):
  order_id       | customer_id    | amount         | segment        | annotation
  ------------------------------------------------------------------------------
  O1             | C1             | 100            | enterprise     | OBSERVED
  O2             | C2             | 250            | smb            | OBSERVED
  O3             | C1             | 900            | enterprise     | OBSERVED

  ✓ Complete answer: fully inside all observability boundaries.


### 0.8 Query crossing orders' retention boundary (t=30)

`t=30 < orders.retention_watermark=40`. Orders AS OF t=30 returns `BEYOND` candidates for every known entity. The join multiplies via `⊗ = min`: `BEYOND ⊗ OBSERVED = BEYOND`. Every output row is tainted — the refusal propagated from orders through the join into the final result **with zero special-case code**.

In [9]:
c30 = customers.as_of(30)
o30 = orders.as_of(30)
joined30 = o30.join(c30, on=["customer_id"])
result30 = split_result(joined30)
result30.report("orders ⋈ customers AS OF t=30  (orders watermark=40, customers watermark=0)")

print()
print("Compare: o30 annotations (what orders.as_of(30) returns):")
o30.show()


  QUERY: orders ⋈ customers AS OF t=30  (orders watermark=40, customers watermark=0)

  ANSWERS (observed, inside retention):
  order_id       | customer_id    | amount         | segment        | annotation
  ------------------------------------------------------------------------------
  (empty)

  REFUSALS (beyond observability boundary):
  order_id       | customer_id    | amount         | segment        | annotation
  ------------------------------------------------------------------------------
  O1             | C1             | 100            | smb            | BEYOND
  O2             | C2             | 250            | smb            | BEYOND
  O3             | C1             | 900            | smb            | BEYOND

  ⚠ HONEST REFUSAL: parts of this answer are unknowable —
    the query's time bound precedes a retention watermark.
    These rows are neither confirmed nor denied.

Compare: o30 annotations (what orders.as_of(30) returns):
  order_id       | customer_id    | a

### 0.9 Projection and `⊕`: "which segments had orders at t=30?"

After the join, project onto `segment`. Multiple rows with the same segment merge via `⊕ = max`. `BEYOND ⊕ BEYOND = BEYOND` — the taint survives projection. A naive engine returns empty (falsely confident); the semiring returns a refusal: *"unknowable, not no."*

In [10]:
seg = o30.join(c30, on=["customer_id"]).project(("segment",))
print("π_segment(orders ⋈ customers) AS OF t=30:")
seg.show()
result_seg = split_result(seg)
print()
if result_seg.refused:
    print("HONEST REFUSAL: segment membership at t=30 is unknowable.")
    print("(BEYOND taint survived both ⊗ in the join and ⊕ in projection.)")
else:
    print("Complete answer (no refusals).")

π_segment(orders ⋈ customers) AS OF t=30:
  segment        | annotation
  ---------------------------
  smb            | BEYOND

HONEST REFUSAL: segment membership at t=30 is unknowable.
(BEYOND taint survived both ⊗ in the join and ⊕ in projection.)


---
## Phase 1: Simulation Grade — Real Logic, Fake Infrastructure

The four load-bearing OWS mechanisms run against simulated storage systems. Real logic, zero real infrastructure dependencies. The point: prove the spec is correct before committing to Delta or Iceberg.

### 1.0 The hash-chained manifest

Append-only JSONL ledger. Each entry commits to its predecessor's SHA-256 hash — making any retroactive edit immediately detectable as a chain break at a specific sequence number.

In [11]:
import hashlib
import json
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable, Optional

class Manifest:
    """Append-only, hash-chained ledger. Each entry commits to its
    predecessor's hash; verify() walks the chain."""

    def __init__(self) -> None:
        self.entries: list[dict] = []

    def append(self, kind: str, **payload) -> dict:
        prev_hash = self.entries[-1]["hash"] if self.entries else "GENESIS"
        body = {"seq": len(self.entries), "kind": kind,
                "prev_hash": prev_hash, **payload}
        body["hash"] = hashlib.sha256(
            json.dumps(body, sort_keys=True, default=str).encode()
        ).hexdigest()[:16]
        self.entries.append(body)
        return body

    def verify(self) -> tuple[bool, Optional[int]]:
        prev = "GENESIS"
        for e in self.entries:
            if e["prev_hash"] != prev:
                return False, e["seq"]
            recomputed = hashlib.sha256(json.dumps(
                {k: v for k, v in e.items() if k != "hash"},
                sort_keys=True, default=str).encode()).hexdigest()[:16]
            if recomputed != e["hash"]:
                return False, e["seq"]
            prev = e["hash"]
        return True, None

In [12]:
MANIFEST = Manifest()

def banner(title: str) -> None:
    print("\n" + "=" * 74)
    print(f"  {title}")
    print("=" * 74)

### 1.1 Example 1: Watermark oracle over a simulated Delta log

Two-phase oracle pattern (the OWS conformance requirement):
1. **Metadata derivation** — scan the log to find the earliest version whose    files all exist in storage.
2. **Empirical validation** — actually attempt reads at `boundary` and    `boundary−1`. Metadata arithmetic is never trusted alone.

The headline dishonesty: a version can still be **listed** in the log while its data files were vacuumed. The oracle measures exactly how many are "ghosts."

In [13]:
# The subtlety the spec insists on (§5): a version can still be LISTED in
# the log while its data files were VACUUMED. The honest boundary is the
# earliest READABLE version, and conformance requires proving it
# empirically, not trusting metadata arithmetic.

@dataclass
class SimulatedDeltaTable:
    name: str
    # version -> (timestamp_day, data_files)
    log: dict[int, tuple[int, set[str]]] = field(default_factory=dict)
    storage: set[str] = field(default_factory=set)   # files that exist

    def commit(self, version: int, day: int, files: set[str]) -> None:
        self.log[version] = (day, files)
        self.storage |= files

    def vacuum(self, retain_after_day: int) -> list[str]:
        """Delete files not referenced by any version at/after the cutoff.
        This is what breaks time travel."""
        current_day = max(d for d, _ in self.log.values())
        live: set[str] = set()
        for v, (day, files) in self.log.items():
            if day >= retain_after_day:
                live |= files
        # files referenced ONLY by pre-cutoff versions get removed
        removed = sorted(self.storage - live)
        self.storage = live
        MANIFEST.append("vacuum", chain=f"delta://{self.name}",
                        cutoff_day=retain_after_day, files_removed=len(removed),
                        recorded_at_day=current_day)
        return removed

    def read_version(self, version: int) -> list[str]:
        """Time travel read. Fails if any required file was vacuumed —
        even though the log entry still exists."""
        if version not in self.log:
            raise LookupError(f"version {version} not in log")
        _, files = self.log[version]
        missing = files - self.storage
        if missing:
            raise FileNotFoundError(
                f"version {version} listed in log but unreadable: "
                f"{len(missing)} data file(s) vacuumed")
        return sorted(files)


def watermark_oracle(t: SimulatedDeltaTable) -> dict:
    """OWS adapter for the simulated Delta table. Two phases:
    (1) derive candidate boundary from metadata,
    (2) EMPIRICALLY validate: boundary readable, boundary-1 not."""
    # Phase 1: metadata derivation — earliest version whose files all exist
    candidate = None
    for v in sorted(t.log):
        _, files = t.log[v]
        if files <= t.storage:
            candidate = v
            break
    # Phase 2: empirical validation (the conformance requirement)
    proof = {}
    try:
        t.read_version(candidate)
        proof["read_at_boundary"] = "SUCCEEDED"
    except Exception as e:
        proof["read_at_boundary"] = f"FAILED ({e})"
    prior = candidate - 1
    if prior in t.log:
        try:
            t.read_version(prior)
            proof["read_below_boundary"] = "SUCCEEDED (VIOLATION!)"
        except FileNotFoundError as e:
            proof["read_below_boundary"] = f"FAILED as expected"
    else:
        proof["read_below_boundary"] = "no prior version in log"
    validated = (proof["read_at_boundary"] == "SUCCEEDED"
                 and "VIOLATION" not in proof["read_below_boundary"])
    wm = {"chain": f"delta://{t.name}",
          "boundary_version": candidate,
          "boundary_day": t.log[candidate][0],
          "evidence_grade": "derived",
          "empirically_validated": validated,
          "proof": proof}
    MANIFEST.append("watermark", **wm)
    return wm


def example_1() -> dict:
    banner("EXAMPLE 1 — Watermark oracle: derive + empirically validate")
    orders = SimulatedDeltaTable("sales/orders")
    # 100 days of daily commits
    for day in range(100):
        orders.commit(version=day, day=day, files={f"part-{day}.parquet"})
    print(f"  Built {len(orders.log)} versions over 100 days.")

    orders.vacuum(retain_after_day=70)
    print(f"  VACUUM ran with 30-day retention (cutoff = day 70).")
    print(f"  Log still lists {len(orders.log)} versions — "
          f"the log LIES about readability.")

    wm = watermark_oracle(orders)
    print(f"\n  Oracle-derived boundary: version {wm['boundary_version']} "
          f"(day {wm['boundary_day']}), grade={wm['evidence_grade']}")
    print(f"  Empirical proof:")
    for k, v in wm["proof"].items():
        print(f"    {k}: {v}")
    print(f"  Conformance: empirically_validated = "
          f"{wm['empirically_validated']}")
    return wm

In [14]:
wm = example_1()


  EXAMPLE 1 — Watermark oracle: derive + empirically validate
  Built 100 versions over 100 days.
  VACUUM ran with 30-day retention (cutoff = day 70).
  Log still lists 100 versions — the log LIES about readability.

  Oracle-derived boundary: version 70 (day 70), grade=derived
  Empirical proof:
    read_at_boundary: SUCCEEDED
    read_below_boundary: FAILED as expected
  Conformance: empirically_validated = True


### 1.2 Example 2: Manifest tamper-evidence

A "malicious admin" rewrites a vacuum entry to hide that files were removed. The hash chain catches it and reports the exact sequence number of the first bad entry.

In [15]:
def example_2() -> None:
    banner("EXAMPLE 2 — Manifest integrity: tampering is detected")
    ok, _ = MANIFEST.verify()
    print(f"  Chain verification after Example 1: {'INTACT' if ok else 'BROKEN'}"
          f"  ({len(MANIFEST.entries)} entries)")

    print("\n  Simulating a malicious retroactive edit: an admin rewrites")
    print("  the vacuum entry to hide that files were removed...")
    victim = next(e for e in MANIFEST.entries if e["kind"] == "vacuum")
    victim["files_removed"] = 0        # the lie

    ok, seq = MANIFEST.verify()
    print(f"  Chain verification now: {'INTACT' if ok else 'BROKEN'}"
          f" — first bad entry at seq={seq}")
    print("  The hash chain converts 'quiet edit' into 'loud, located breach.'")
    # restore truth for the rest of the demo
    victim["files_removed"] = 70
    victim["hash"] = hashlib.sha256(json.dumps(
        {k: v for k, v in victim.items() if k != "hash"},
        sort_keys=True, default=str).encode()).hexdigest()[:16]
    # fix forward links
    for i in range(victim["seq"] + 1, len(MANIFEST.entries)):
        MANIFEST.entries[i]["prev_hash"] = MANIFEST.entries[i - 1]["hash"]
        e = MANIFEST.entries[i]
        e["hash"] = hashlib.sha256(json.dumps(
            {k: v for k, v in e.items() if k != "hash"},
            sort_keys=True, default=str).encode()).hexdigest()[:16]

In [16]:
example_2()


  EXAMPLE 2 — Manifest integrity: tampering is detected
  Chain verification after Example 1: INTACT  (2 entries)

  Simulating a malicious retroactive edit: an admin rewrites
  the vacuum entry to hide that files were removed...
  Chain verification now: BROKEN — first bad entry at seq=0
  The hash chain converts 'quiet edit' into 'loud, located breach.'


### 1.3 Example 3: The contradicted watermark

A DBA silently shrinks WAL retention from 14 days to 3 days during a cost-cutting sprint. The OWS heartbeat re-introspects the source, detects the contradiction, records it in the manifest, and automatically tightens all downstream verdicts to the conservative boundary. **This is the alert nothing on the market sends today.**

In [17]:
@dataclass
class SimulatedPostgres:
    """Stands in for catalog introspection: the source SELF-REPORTS its
    retention posture; we never parse the WAL."""
    wal_retention_days: int
    slot_restart_lag_days: int   # how far back the replication slot reaches

    def introspect(self) -> dict:
        # boundary is limited by the MOST restrictive artifact
        limiting = ("replication_slot"
                    if self.slot_restart_lag_days < self.wal_retention_days
                    else "wal_retention")
        return {
            "wal_retention_days": self.wal_retention_days,
            "slot_restart_lag_days": self.slot_restart_lag_days,
            "boundary_days_back": min(self.wal_retention_days,
                                      self.slot_restart_lag_days),
            "limited_by": limiting,
        }


def example_3() -> dict:
    banner("EXAMPLE 3 — Contradicted watermark: silent retention change")
    pg = SimulatedPostgres(wal_retention_days=14, slot_restart_lag_days=14)

    # Setup: introspect -> suggest -> human countersigns
    report = pg.introspect()
    print(f"  Setup introspection: boundary reaches {report['boundary_days_back']} "
          f"days back (limited by: {report['limited_by']})")
    confirmed = MANIFEST.append(
        "countersignature", chain="pg://crm/customers",
        confirmed_boundary_days_back=report["boundary_days_back"],
        raw_introspection=report,
        identity="zach@company.example", via="sso")
    print(f"  Countersigned by {confirmed['identity']} — grade is now "
          f"derived-countersigned.")

    # Months later: a DBA shrinks WAL retention in a cost-cutting sprint.
    pg.wal_retention_days = 3
    print("\n  ...weeks pass. A DBA quietly sets WAL retention 14d -> 3d...")

    # Heartbeat re-introspection
    fresh = pg.introspect()
    confirmed_days = confirmed["confirmed_boundary_days_back"]
    if fresh["boundary_days_back"] < confirmed_days:
        MANIFEST.append("contradiction", chain="pg://crm/customers",
                        confirmed_days_back=confirmed_days,
                        observed_days_back=fresh["boundary_days_back"],
                        limited_by=fresh["limited_by"])
        effective = fresh["boundary_days_back"]  # conservative boundary
        print(f"  HEARTBEAT ALERT — contradicted watermark:")
        print(f"    confirmed: {confirmed_days} days back (signed)")
        print(f"    observed now: {fresh['boundary_days_back']} days back "
              f"(limited by {fresh['limited_by']})")
        print(f"  Verdicts tighten to the CONSERVATIVE boundary "
              f"({effective} days) until re-confirmation.")
        print(f"  This is the alert nothing on the market sends today:")
        print(f"  'your source no longer supports your audit obligations.'")
    return {"chain": "pg://crm/customers",
            "effective_days_back": fresh["boundary_days_back"],
            "state": "contradicted"}

In [18]:
pg_wm = example_3()


  EXAMPLE 3 — Contradicted watermark: silent retention change
  Setup introspection: boundary reaches 14 days back (limited by: wal_retention)
  Countersigned by zach@company.example — grade is now derived-countersigned.

  ...weeks pass. A DBA quietly sets WAL retention 14d -> 3d...
  HEARTBEAT ALERT — contradicted watermark:
    confirmed: 14 days back (signed)
    observed now: 3 days back (limited by wal_retention)
  Verdicts tighten to the CONSERVATIVE boundary (3 days) until re-confirmation.
  This is the alert nothing on the market sends today:
  'your source no longer supports your audit obligations.'


### 1.4 Example 4: The verdict engine

Five queries, five different verdict patterns:
- **A** — monotone aggregate fully inside boundaries → `EXACT`
- **B** — monotone aggregate crossing the gap → `BOUNDED` (lower bound known, upper not)
- **C** — same query but pre-vacuum escrow snapshots exist → tighter `BOUNDED`
- **D** — non-monotone (customers with NO orders) across gap → `REFUSED`   + monotone complement offered
- **E** — downstream gold table, materialization lag → `BOUNDED`   (twice-temporal correction, spec §6.2)

In [19]:
class Verdict(Enum):
    EXACT = "EXACT"
    BOUNDED = "BOUNDED"
    REFUSED = "REFUSED"


@dataclass
class ChainState:
    chain: str
    boundary_day: int          # earliest honestly queryable day
    grade: str
    # optional escrowed aggregates for gap days: day -> {"revenue": x, "orders": n}
    escrow: dict[int, dict] = field(default_factory=dict)


def render(v: Verdict, detail: str, limiting: Optional[str]) -> None:
    mark = {"EXACT": "✓", "BOUNDED": "≈", "REFUSED": "⊘"}[v.value]
    print(f"  {mark} VERDICT: {v.value}")
    print(f"    {detail}")
    if limiting:
        print(f"    limiting link: {limiting}")


def example_4(orders_wm: dict) -> None:
    banner("EXAMPLE 4 — Verdict engine: EXACT / BOUNDED / REFUSED")

    # Chain states as an Alethe deployment would hold them
    orders = ChainState("delta://sales/orders",
                        boundary_day=orders_wm["boundary_day"],
                        grade="derived")
    customers = ChainState("delta://crm/customers", boundary_day=0,
                           grade="derived")

    # Observed data (inside orders' boundary): day -> (revenue, n_orders)
    observed = {d: (1000 + 10 * d, 5) for d in range(70, 100)}
    # Candidate knowledge for the vacuumed gap (row-shapes known from
    # surviving metadata; states unknowable) — 70 days, ~5 orders/day
    candidate_days = list(range(0, 70))

    print(f"  Chains: orders boundary=day {orders.boundary_day} (vacuumed "
          f"before), customers boundary=day {customers.boundary_day}\n")

    # --- Query A: monotone aggregate fully inside boundaries -> EXACT ---
    print("  QUERY A: SUM(revenue) for days 80-99  [monotone, all green]")
    total = sum(observed[d][0] for d in range(80, 100))
    render(Verdict.EXACT, f"revenue = {total:,} — every chain inside "
           f"boundary; grade=derived on all paths", None)

    # --- Query B: monotone aggregate crossing the boundary -> BOUNDED ---
    print("\n  QUERY B: SUM(revenue) for days 0-99  [monotone, crosses gap]")
    at_least = sum(v[0] for v in observed.values())
    render(Verdict.BOUNDED,
           f"revenue in [{at_least:,}, UNBOUNDED) — at_least from observed "
           f"days 70-99; days 0-69 are BEYOND (upper bound unknowable: "
           f"population destroyed)",
           f"{orders.chain} boundary=day {orders.boundary_day}")

    # --- Query C: same, but escrow exists -> tighter BOUNDED ---
    print("\n  QUERY C: same query, but pre-vacuum ESCROW snapshots exist")
    orders.escrow = {d: {"revenue": 1000 + 10 * d} for d in range(0, 70)}
    escrowed = sum(e["revenue"] for e in orders.escrow.values())
    render(Verdict.BOUNDED,
           f"revenue = {at_least + escrowed:,} at day-level granularity — "
           f"days 0-69 answered from escrowed aggregates "
           f"(grade=escrowed, weaker than derived; row detail still BEYOND)",
           f"{orders.chain} gap disposition=escrowed")

    # --- Query D: non-monotone across the gap -> REFUSED + complement ---
    print("\n  QUERY D: customers with NO orders, days 0-99  [NON-monotone]")
    render(Verdict.REFUSED,
           "absence of order-evidence in days 0-69 is not evidence of "
           "absence — no safe lower bound exists for a negation across "
           "a BEYOND interval",
           f"{orders.chain} boundary=day {orders.boundary_day}")
    print("    offered monotone complement: customers WITH orders in "
          "days 70-99 -> EXACT")

    # --- Query E: twice-temporal correction on a downstream gold table ---
    print("\n  QUERY E: gold.daily_revenue AS OF day 99  [materialization lag]")
    last_materialization_day = 97   # nightly job; last successful run day 97
    print(f"    gold last materialized: day {last_materialization_day} "
          f"(consumed upstream state as of day {last_materialization_day})")
    render(Verdict.BOUNDED,
           "requested day 99, but gold reflects upstream only through day "
           "97 — result is exact AS OF day 97, stale for days 98-99; "
           "certifying it as day-99 state would be the subtle dishonesty "
           "OWS exists to kill (twice-temporal correction, spec §6.2)",
           "materialization lag on edge orders->gold.daily_revenue")

    MANIFEST.append("verdict-demo-complete", queries=5)

In [20]:
example_4(wm)


  EXAMPLE 4 — Verdict engine: EXACT / BOUNDED / REFUSED
  Chains: orders boundary=day 70 (vacuumed before), customers boundary=day 0

  QUERY A: SUM(revenue) for days 80-99  [monotone, all green]
  ✓ VERDICT: EXACT
    revenue = 37,900 — every chain inside boundary; grade=derived on all paths

  QUERY B: SUM(revenue) for days 0-99  [monotone, crosses gap]
  ≈ VERDICT: BOUNDED
    revenue in [55,350, UNBOUNDED) — at_least from observed days 70-99; days 0-69 are BEYOND (upper bound unknowable: population destroyed)
    limiting link: delta://sales/orders boundary=day 70

  QUERY C: same query, but pre-vacuum ESCROW snapshots exist
  ≈ VERDICT: BOUNDED
    revenue = 149,500 at day-level granularity — days 0-69 answered from escrowed aggregates (grade=escrowed, weaker than derived; row detail still BEYOND)
    limiting link: delta://sales/orders gap disposition=escrowed

  QUERY D: customers with NO orders, days 0-99  [NON-monotone]
  ⊘ VERDICT: REFUSED
    absence of order-evidence in da

### 1.5 Final manifest state after all four examples

In [21]:
banner("MANIFEST — final state")
ok, _ = MANIFEST.verify()
print(f"  {len(MANIFEST.entries)} entries, chain {'INTACT' if ok else 'BROKEN'}. Entry kinds:")
kinds: dict[str, int] = {}
for e in MANIFEST.entries:
    kinds[e["kind"]] = kinds.get(e["kind"], 0) + 1
for k, n in kinds.items():
    print(f"    {k}: {n}")
print()
print("Last 3 entries:")
for e in MANIFEST.entries[-3:]:
    print(f"  seq={e['seq']} kind={e['kind']} hash={e['hash']} prev={e['prev_hash']}")


  MANIFEST — final state
  5 entries, chain INTACT. Entry kinds:
    vacuum: 1
    watermark: 1
    countersignature: 1
    contradiction: 1
    verdict-demo-complete: 1

Last 3 entries:
  seq=2 kind=countersignature hash=5bdeae8abb66870a prev=0a20bdbf19367b91
  seq=3 kind=contradiction hash=c8b66e91618d2543 prev=5bdeae8abb66870a
  seq=4 kind=verdict-demo-complete hash=fde1ba9565a6ac28 prev=c8b66e91618d2543


---
## Phase 2: Real Delta Lake Oracle

No simulation. This phase creates a real Delta table on disk, runs a real `VACUUM (retention_hours=0)`, then proves the boundary empirically via real time-travel reads.

**The headline dishonesty**: after VACUUM, the `_delta_log` still *lists* every historical version — but most are physically unreadable. A naive `AS OF version N` silently fails at read time with no warning at plan time. The oracle turns that into a provable, manifest-ready boundary.

### 2.0 Setup: imports and table path

In [22]:
from __future__ import annotations
import json
import os
from pathlib import Path

import pandas as pd
from deltalake import DeltaTable, write_deltalake

TABLE = Path(".") / "lakehouse" / "sales_orders"

### 2.1 Build a real Delta table with 8 versions

Each write uses `mode="overwrite"` so prior parquet files become unreferenced but the `_delta_log` still contains entries for them. This is the exact setup that makes AS OF queries silently unreliable.

In [23]:
def build_table() -> None:
    TABLE.parent.mkdir(parents=True, exist_ok=True)
    for v in range(8):
        df = pd.DataFrame({
            "order_id": [f"O{v}{i}" for i in range(5)],
            "customer_id": [f"C{i % 3}" for i in range(5)],
            "amount": [100.0 * (v + 1) + i for i in range(5)],
            "day": [v * 4] * 5,
        })
        # overwrite each time so prior files become unreferenced
        write_deltalake(TABLE, df, mode="overwrite")
    print(f"  Wrote 8 real versions to {TABLE}")

In [24]:
import shutil
if TABLE.exists():
    shutil.rmtree(TABLE)   # clean slate for re-runs
build_table()

  Wrote 8 real versions to lakehouse/sales_orders


### 2.2 Run VACUUM (retention_hours=0)

This is the footgun every engine exposes and OWS exists to make honest. Setting `retention_hours=0` with enforcement disabled is exactly what a cost-control script or aggressive data team would do.

In [25]:
def vacuum_table() -> list[str]:
    dt = DeltaTable(str(TABLE))
    removed = dt.vacuum(retention_hours=0,
                        enforce_retention_duration=False,
                        dry_run=False)
    print(f"  REAL VACUUM removed {len(removed)} parquet file(s).")
    return removed

In [26]:
removed_files = vacuum_table()
print(f"Files removed: {removed_files[:3]}{'...' if len(removed_files) > 3 else ''}")

  REAL VACUUM removed 7 parquet file(s).
Files removed: ['part-00000-50d50746-0b3a-4cf0-9f25-2ec6a06c3a12-c000.snappy.parquet', 'part-00000-1d36cb9b-4e13-4619-86ac-eaf2d5048cb7-c000.snappy.parquet', 'part-00000-8e029143-6c87-41a5-bf83-2fce28d6b513-c000.snappy.parquet']...


### 2.3 Oracle Phase A: metadata derivation from the raw `_delta_log`

Replay every `add`/`remove` action in the commit JSON files to reconstruct each version's exact file set. Then check physical file existence. The candidate boundary is the earliest version whose full file set still exists.

Note: the log's `history()` method will **still list** all versions — that's the lie this phase is measuring.

In [27]:
def replay_log(table: Path) -> dict[int, set[str]]:
    """Reconstruct each version's full file set by replaying add/remove
    actions from the commit JSONs. Returns {version: {relative paths}}."""
    log_dir = table / "_delta_log"
    commits = sorted(p for p in log_dir.glob("*.json"))
    live: set[str] = set()
    per_version: dict[int, set[str]] = {}
    for commit in commits:
        version = int(commit.stem)
        for line in commit.read_text().splitlines():
            action = json.loads(line)
            if "add" in action:
                live.add(action["add"]["path"])
            elif "remove" in action:
                live.discard(action["remove"]["path"])
        per_version[version] = set(live)
    return per_version


def derive_boundary(table: Path) -> tuple[int | None, dict[int, dict]]:
    per_version = replay_log(table)
    detail: dict[int, dict] = {}
    candidate = None
    for v in sorted(per_version):
        files = per_version[v]
        missing = {f for f in files if not (table / f).exists()}
        detail[v] = {"files": len(files), "missing": len(missing)}
        if not missing and candidate is None:
            candidate = v
    return candidate, detail

In [28]:
from deltalake import DeltaTable

candidate, detail = derive_boundary(TABLE)
dt_now = DeltaTable(str(TABLE))
listed = [h["version"] for h in dt_now.history()]

print(f"Log LISTS {len(listed)} versions: {sorted(listed)}")
print()
for v in sorted(detail):
    d = detail[v]
    flag = "  ← candidate boundary" if v == candidate else ""
    print(f"  v{v}: {d['files']} file(s), {d['missing']} missing{flag}")
print()
print(f"Derived candidate boundary: version {candidate}")

Log LISTS 10 versions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

  v0: 1 file(s), 1 missing
  v1: 1 file(s), 1 missing
  v2: 1 file(s), 1 missing
  v3: 1 file(s), 1 missing
  v4: 1 file(s), 1 missing
  v5: 1 file(s), 1 missing
  v6: 1 file(s), 1 missing
  v7: 1 file(s), 0 missing  ← candidate boundary
  v8: 1 file(s), 0 missing
  v9: 1 file(s), 0 missing

Derived candidate boundary: version 7


### 2.4 Oracle Phase B: empirical validation (real time-travel reads)

The spec conformance requirement: actually attempt a time-travel read at the candidate boundary and at `boundary−1`. Metadata arithmetic is never trusted alone. The boundary is confirmed iff reads at/after it succeed and reads before it fail.

In [29]:
def try_read(table: Path, version: int) -> tuple[bool, str]:
    try:
        dt = DeltaTable(str(table), version=version)
        n = dt.to_pyarrow_table().num_rows   # force real file reads
        return True, f"read {n} rows"
    except Exception as e:
        return False, type(e).__name__


def empirically_validate(table: Path, candidate: int,
                         versions: list[int]) -> tuple[bool, dict[int, str]]:
    results: dict[int, str] = {}
    ok = True
    for v in versions:
        readable, note = try_read(table, v)
        results[v] = f"{'READABLE' if readable else 'UNREADABLE'} ({note})"
        if v >= candidate and not readable:
            ok = False      # boundary claims readability it can't deliver
        if v < candidate and readable:
            ok = False      # boundary is too conservative — also a defect
    return ok, results

In [30]:
versions = sorted(detail)
validated, results = empirically_validate(TABLE, candidate, versions)

print("Time-travel read results:")
for v in versions:
    print(f"  AS OF version {v}: {results[v]}")
print()
print(f"empirically_validated = {validated}")

unreadable = [v for v in versions if results[v].startswith("UNREADABLE")]
print(f"\nThe dishonesty, measured: log advertises {len(listed)} versions; "
      f"only {len(versions) - len(unreadable)} are actually readable.")
print(f"Ghost versions (listed but unreadable): {unreadable}")

Time-travel read results:
  AS OF version 0: UNREADABLE (FileNotFoundError)
  AS OF version 1: UNREADABLE (FileNotFoundError)
  AS OF version 2: UNREADABLE (FileNotFoundError)
  AS OF version 3: UNREADABLE (FileNotFoundError)
  AS OF version 4: UNREADABLE (FileNotFoundError)
  AS OF version 5: UNREADABLE (FileNotFoundError)
  AS OF version 6: UNREADABLE (FileNotFoundError)
  AS OF version 7: READABLE (read 5 rows)
  AS OF version 8: READABLE (read 5 rows)
  AS OF version 9: READABLE (read 5 rows)

empirically_validated = True

The dishonesty, measured: log advertises 10 versions; only 3 are actually readable.
Ghost versions (listed but unreadable): [0, 1, 2, 3, 4, 5, 6]


### 2.5 Emit the watermark tuple

The completed watermark: `(boundary, evidence_grade, empirically_validated, proof)`. This is the structure that gets written to the hash-chained manifest in Phase 3.

In [31]:
import json

watermark = {
    "chain": f"delta://{TABLE.name}",
    "boundary": {"version": candidate},
    "evidence_grade": "derived",
    "empirically_validated": validated,
    "proof": {
        "listed_versions": sorted(int(x) for x in listed),
        "readable_versions": [v for v in versions if results[v].startswith("READABLE")],
    },
}
print(json.dumps(watermark, indent=2))

{
  "chain": "delta://sales_orders",
  "boundary": {
    "version": 7
  },
  "evidence_grade": "derived",
  "empirically_validated": true,
  "proof": {
    "listed_versions": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9
    ],
    "readable_versions": [
      7,
      8,
      9
    ]
  }
}


---
## Phase 3: Manifest + Iceberg Second Adapter

Two steps:

**Step 1** — Wire the Delta oracle into a persisted hash-chained manifest. Two independent boundary derivations are cross-checked: file-existence replay vs. vacuum-commit records already in the `_delta_log` itself. Corroboration upgrades confidence.

**Step 2** — The same watermark contract against a real Apache Iceberg table (pyiceberg + sqlite catalog). History destroyed out-of-band simulates the real-world `remove_orphan_files` / lifecycle-policy failure mode.

**Key Iceberg finding** (discovered empirically, not by reading the spec): readability is **not monotone** in sequence number. Delete/append pairs leave empty intermediate snapshots readable between destroyed ones. The honest boundary is a *suffix* claim plus readable islands in `proof`.

### 3.0 Disk-backed hash-chained manifest

Same hash-chain logic as Phase 1, but persisted to `ows_manifest.jsonl`. Existing entries are loaded on construction so the ledger is append-only across runs.

In [32]:
import hashlib
import json
import shutil
from pathlib import Path
import pyarrow as pa

class Manifest:
    def __init__(self, path: Path):
        self.path = path
        self.entries: list[dict] = []
        if path.exists():
            self.entries = [json.loads(l) for l in path.read_text().splitlines()]

    def append(self, kind: str, **payload) -> dict:
        prev_hash = self.entries[-1]["hash"] if self.entries else "GENESIS"
        body = {"seq": len(self.entries), "kind": kind,
                "prev_hash": prev_hash, **payload}
        body["hash"] = hashlib.sha256(json.dumps(
            body, sort_keys=True, default=str).encode()).hexdigest()[:16]
        self.entries.append(body)
        self.path.write_text("\n".join(
            json.dumps(e, default=str) for e in self.entries))
        return body

    def verify(self) -> bool:
        prev = "GENESIS"
        for e in self.entries:
            if e["prev_hash"] != prev:
                return False
            body = {k: v for k, v in e.items() if k != "hash"}
            if hashlib.sha256(json.dumps(body, sort_keys=True,
                                         default=str).encode()
                              ).hexdigest()[:16] != e["hash"]:
                return False
            prev = e["hash"]
        return True

### 3.1 Step 1: Delta oracle → manifest with cross-check

Two independent derivations of the same boundary:
- **Derivation A** (file existence): same replay as Phase 2.
- **Derivation B** (vacuum commits in log): find `VACUUM END` operations in   `history()` and infer the surviving boundary from the last pre-vacuum write.

Corroboration (A == B) is recorded in the manifest entry. Two independent methods reaching the same answer is stronger evidence than either alone.

In [33]:
def delta_watermark_to_manifest(table: Path, manifest: Manifest) -> dict:
    from deltalake import DeltaTable

    # --- derivation A: file-existence replay (from ows_delta_oracle) ---
    log_dir = table / "_delta_log"
    live: set[str] = set()
    per_version: dict[int, set[str]] = {}
    for commit in sorted(log_dir.glob("*.json")):
        for line in commit.read_text().splitlines():
            a = json.loads(line)
            if "add" in a:
                live.add(a["add"]["path"])
            elif "remove" in a:
                live.discard(a["remove"]["path"])
        per_version[int(commit.stem)] = set(live)
    boundary_by_files = next(
        v for v in sorted(per_version)
        if all((table / f).exists() for f in per_version[v]))

    # --- derivation B: the log's own VACUUM commits ---
    dt = DeltaTable(str(table))
    vacuum_ends = [h for h in dt.history() if h.get("operation") == "VACUUM END"]
    # after a vacuum, the earliest surviving version is the earliest WRITE
    # whose files were still referenced at vacuum time; with full
    # overwrites that is the last WRITE before the first vacuum commit.
    if vacuum_ends:
        last_vacuum_version = max(h["version"] for h in vacuum_ends)
        writes_before = [h["version"] for h in dt.history()
                         if h.get("operation") == "WRITE"
                         and h["version"] < last_vacuum_version]
        boundary_by_log = max(writes_before) if writes_before else None
    else:
        boundary_by_log = min(per_version)

    corroborated = (boundary_by_log == boundary_by_files)

    # --- empirical validation at the agreed boundary ---
    def readable(v: int) -> bool:
        try:
            DeltaTable(str(table), version=v).to_pyarrow_table()
            return True
        except Exception:
            return False

    validated = readable(boundary_by_files) and (
        boundary_by_files - 1 not in per_version
        or not readable(boundary_by_files - 1))

    entry = manifest.append(
        "watermark",
        chain=f"delta://{table.name}",
        boundary={"version": boundary_by_files},
        evidence_grade="derived",
        empirically_validated=validated,
        proof={
            "derivation_file_existence": boundary_by_files,
            "derivation_vacuum_commits": boundary_by_log,
            "corroborated": corroborated,
            "vacuum_end_versions": [h["version"] for h in vacuum_ends],
        })
    return entry

In [34]:
manifest = Manifest(Path(".") / "ows_manifest.jsonl")
delta_table = Path(".") / "lakehouse" / "sales_orders"

entry = delta_watermark_to_manifest(delta_table, manifest)
print(f"boundary: version {entry['boundary']['version']}")
print(f"derivation A (file existence): v{entry['proof']['derivation_file_existence']}")
print(f"derivation B (vacuum commits in log): v{entry['proof']['derivation_vacuum_commits']}")
print(f"corroborated: {entry['proof']['corroborated']}   "
      f"empirically_validated: {entry['empirically_validated']}")
print(f"manifest entry seq={entry['seq']} hash={entry['hash']}")

boundary: version 7
derivation A (file existence): v7
derivation B (vacuum commits in log): v7
corroborated: True   empirically_validated: True
manifest entry seq=0 hash=a242f81baebdbf3f


### 3.2 Build a real Iceberg table (5 snapshots)

Each overwrite makes the previous snapshot's data files unreferenced (snapshot metadata still lists them). This is the same pattern as Delta VACUUM — same problem, different metadata model.

In [35]:
WAREHOUSE = Path(".") / "iceberg_warehouse"


def build_iceberg_table():
    from pyiceberg.catalog.sql import SqlCatalog
    if WAREHOUSE.exists():
        shutil.rmtree(WAREHOUSE)
    WAREHOUSE.mkdir(parents=True)
    catalog = SqlCatalog(
        "local",
        uri=f"sqlite:///{WAREHOUSE}/catalog.db",
        warehouse=f"file://{WAREHOUSE}")
    catalog.create_namespace("sales")
    schema = pa.schema([
        ("order_id", pa.string()),
        ("customer_id", pa.string()),
        ("amount", pa.float64()),
    ])
    tbl = catalog.create_table("sales.orders", schema=schema)
    # 5 snapshots, each an overwrite so old files become unreferenced
    for s in range(5):
        batch = pa.table({
            "order_id": [f"O{s}{i}" for i in range(4)],
            "customer_id": [f"C{i % 2}" for i in range(4)],
            "amount": [100.0 * (s + 1) + i for i in range(4)],
        })
        tbl.overwrite(batch)
    return catalog.load_table("sales.orders")

In [36]:
tbl = build_iceberg_table()
print(f"Built real Iceberg table with {len(tbl.metadata.snapshots)} snapshots.")
print(f"Snapshot sequence numbers: "
      f"{[s.sequence_number for s in sorted(tbl.metadata.snapshots, key=lambda s: s.sequence_number)]}")

Built real Iceberg table with 9 snapshots.
Snapshot sequence numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyiceberg/table/__init__.py:639: UserWarning: Delete operation did not match any records
  self.delete(


### 3.3 Destroy old history out-of-band

Simulate the `remove_orphan_files` / aggressive lifecycle-policy failure mode: data files for old snapshots are deleted directly, but snapshot metadata still lists every snapshot. **Same lie, different format.**

In [37]:
def destroy_old_history(tbl, keep_last_n: int = 2) -> int:
    """Simulate the real-world failure mode: old data files deleted
    out-of-band (aggressive remove_orphan_files, lifecycle policy on the
    bucket, or a well-meaning cleanup script). Snapshot metadata still
    lists everything; the files are gone."""
    snapshots = sorted(tbl.metadata.snapshots, key=lambda s: s.sequence_number)
    doomed = snapshots[:-keep_last_n]
    removed = 0
    keep_files: set[str] = set()
    for snap in snapshots[-keep_last_n:]:
        for task in tbl.scan(snapshot_id=snap.snapshot_id).plan_files():
            keep_files.add(task.file.file_path)
    for snap in doomed:
        try:
            for task in tbl.scan(snapshot_id=snap.snapshot_id).plan_files():
                p = Path(task.file.file_path.replace("file://", ""))
                if task.file.file_path not in keep_files and p.exists():
                    p.unlink()
                    removed += 1
        except Exception:
            pass  # already unreadable
    return removed

In [38]:
n_snaps = len(tbl.metadata.snapshots)
removed = destroy_old_history(tbl, keep_last_n=2)
print(f"Snapshot metadata still lists {n_snaps} snapshots.")
print(f"Data files deleted: {removed}")
print(f"The metadata is now lying in exactly the same way Delta's log did.")

Snapshot metadata still lists 9 snapshots.
Data files deleted: 4
The metadata is now lying in exactly the same way Delta's log did.


### 3.4 Iceberg oracle: same contract, different metadata model

Phase A and B are combined: probe every snapshot with a real read, then derive the boundary under **suffix semantics** — the earliest snapshot from which *all later* snapshots are readable.

In [39]:
def iceberg_watermark_to_manifest(tbl, manifest: Manifest) -> dict:
    """Same contract as the Delta adapter: derive boundary from snapshot
    lineage + file existence, then validate with real reads."""
    snapshots = sorted(tbl.metadata.snapshots, key=lambda s: s.sequence_number)

    def snapshot_readable(snap) -> tuple[bool, str]:
        try:
            n = tbl.scan(snapshot_id=snap.snapshot_id).to_arrow().num_rows
            return True, f"read {n} rows"
        except Exception as e:
            return False, type(e).__name__

    # Phase A+B combined: probe every snapshot with a REAL read, then
    # derive the boundary under SUFFIX semantics — the earliest snapshot
    # from which ALL later snapshots are readable. (Iceberg falsified
    # the naive single-point model: delete/append pairs create empty
    # intermediate snapshots that stay trivially readable between
    # destroyed ones. Readability is not monotone in sequence number,
    # so the honest claim is a suffix boundary PLUS the readable
    # islands below it, recorded as proof.)
    probes = []
    for snap in snapshots:
        ok, note = snapshot_readable(snap)
        probes.append((snap, ok, note))

    candidate = None
    for i, (snap, ok, _) in enumerate(probes):
        if all(ok2 for _, ok2, _ in probes[i:]):
            candidate = snap
            break

    results = {str(s.snapshot_id): f"{'READABLE' if ok else 'UNREADABLE'} ({n})"
               for s, ok, n in probes}
    islands = [str(s.snapshot_id) for s, ok, _ in probes
               if ok and s.sequence_number < candidate.sequence_number]
    validated = candidate is not None and all(
        ok for s, ok, _ in probes
        if s.sequence_number >= candidate.sequence_number)

    entry = manifest.append(
        "watermark",
        chain="iceberg://sales.orders",
        boundary={"snapshot_id": str(candidate.snapshot_id),
                  "sequence_number": candidate.sequence_number,
                  "timestamp_ms": candidate.timestamp_ms},
        evidence_grade="derived",
        empirically_validated=validated,
        proof={"snapshots_listed": len(snapshots),
               "snapshots_readable": sum(
                   1 for r in results.values() if r.startswith("READABLE")),
               "readable_islands_below_boundary": islands,
               "per_snapshot": results},
        note=("Iceberg's immutable snapshot IDs make better audit anchors "
              "than timestamps — the boundary cites an ID, not a clock."))
    return entry

In [40]:
entry2 = iceberg_watermark_to_manifest(tbl, manifest)
print(f"boundary: snapshot {entry2['boundary']['snapshot_id'][:12]}… "
      f"(seq {entry2['boundary']['sequence_number']})")
print(f"snapshots listed: {entry2['proof']['snapshots_listed']}, "
      f"readable: {entry2['proof']['snapshots_readable']}")
print()
for sid, r in entry2["proof"]["per_snapshot"].items():
    print(f"  {sid[:12]}…: {r}")
print()
islands = entry2["proof"]["readable_islands_below_boundary"]
print(f"Readable islands below boundary: {len(islands)} "
      f"(SUFFIX semantics — not monotone in sequence number)")
print(f"empirically_validated: {entry2['empirically_validated']}")

boundary: snapshot 681226287574… (seq 8)
snapshots listed: 9, readable: 5

  451369671659…: UNREADABLE (FileNotFoundError)
  600540481735…: READABLE (read 0 rows)
  452690976627…: UNREADABLE (FileNotFoundError)
  328050948939…: READABLE (read 0 rows)
  125150574737…: UNREADABLE (FileNotFoundError)
  702586537875…: READABLE (read 0 rows)
  410743729181…: UNREADABLE (FileNotFoundError)
  681226287574…: READABLE (read 0 rows)
  323840413486…: READABLE (read 4 rows)

Readable islands below boundary: 3 (SUFFIX semantics — not monotone in sequence number)
empirically_validated: True


### 3.5 Final manifest state

Two real adapters, two real table formats, one hash-chained ledger. The chain integrity check confirms neither entry was tampered with after writing.

In [41]:
print(f"{len(manifest.entries)} entries, chain "
      f"{'INTACT' if manifest.verify() else 'BROKEN'} — "
      f"two formats, one contract, one ledger.")
print(f"persisted at {manifest.path}")
print()
for e in manifest.entries:
    print(f"  seq={e['seq']} kind={e['kind']} "
          f"chain={e.get('chain', 'n/a')} "
          f"hash={e['hash']}")

2 entries, chain INTACT — two formats, one contract, one ledger.
persisted at ows_manifest.jsonl

  seq=0 kind=watermark chain=delta://sales_orders hash=a242f81baebdbf3f
  seq=1 kind=watermark chain=iceberg://sales.orders hash=1d3218071b476bea
